# Exploratory Data Analysis: CUBIT-Seg vs. DeepCrack

This notebook inspects the dataset structure and domain characteristics before modeling:
- **Source Domain (`DeepCrack`)**: 537 images (road/concrete close-ups)
- **Target Evaluation Domain (`CUBIT-Seg`)**: 5,462 images (UAV/DSLR building facades)

Key objectives:
1. Verify image counts, resolutions, and empty-mask ratios.
2. Quantify foreground crack pixel ratios (class imbalance) across both datasets.
3. Visually inspect representative RGB images and ground-truth binary masks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data import CrackDataset, load_binary_mask
from src.paths import ASSETS_DIR, DATA_DIR

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
cubit_ds = CrackDataset(DATA_DIR / "cubit_seg", dataset_type="cubit_seg")
dc_train = CrackDataset(DATA_DIR / "deepcrack", dataset_type="deepcrack", split="train")
dc_test = CrackDataset(DATA_DIR / "deepcrack", dataset_type="deepcrack", split="test")

print(f"CUBIT-Seg total : {len(cubit_ds)}")
print(f"DeepCrack total : {len(dc_train) + len(dc_test)} (train={len(dc_train)}, test={len(dc_test)})")

In [ ]:
def collect_stats(dataset: CrackDataset, name: str) -> pd.DataFrame:
    records = []
    for pair in tqdm(dataset.pairs, desc=f"Scanning {name}"):
        mask = load_binary_mask(pair.mask_path)
        h, w = mask.shape
        crack_pixels = int(mask.sum())
        crack_ratio = crack_pixels / (h * w) * 100.0

        records.append(
            {
                "dataset": name,
                "subset": pair.subset,
                "sample_id": pair.sample_id,
                "height": h,
                "width": w,
                "crack_pixels": crack_pixels,
                "crack_ratio_pct": crack_ratio,
                "is_empty": crack_pixels == 0,
            }
        )
    return pd.DataFrame(records)


df_cubit = collect_stats(cubit_ds, "CUBIT-Seg")
df_dc_train = collect_stats(dc_train, "DeepCrack")
df_dc_test = collect_stats(dc_test, "DeepCrack")
df_all = pd.concat([df_cubit, df_dc_train, df_dc_test], ignore_index=True)

summary = (
    df_all.groupby("dataset")
    .agg(
        num_images=("sample_id", "count"),
        empty_masks=("is_empty", "sum"),
        mean_crack_ratio_pct=("crack_ratio_pct", "mean"),
        median_crack_ratio_pct=("crack_ratio_pct", "median"),
        p95_crack_ratio_pct=("crack_ratio_pct", lambda s: np.percentile(s, 95)),
    )
    .round(3)
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, color in [("CUBIT-Seg", "#2563eb"), ("DeepCrack", "#ea580c")]:
    sub = df_all[df_all["dataset"] == name]
    axes[0].hist(
        sub["crack_ratio_pct"],
        bins=40,
        range=(0, 15),
        density=True,
        alpha=0.55,
        label=f"{name} (n={len(sub)}, median={sub['crack_ratio_pct'].median():.2f}%)",
        color=color,
    )

axes[0].set_title("Crack Area Ratio Distribution (%)")
axes[0].set_xlabel("Crack Area Ratio (%)")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].grid(alpha=0.25)

subset_counts = df_all.groupby(["dataset", "subset"]).size().reset_index(name="count")
labels = [f"{r.dataset}\n({r.subset})" for _, r in subset_counts.iterrows()]
colors = ["#2563eb" if r.dataset == "CUBIT-Seg" else "#ea580c" for _, r in subset_counts.iterrows()]
bars = axes[1].bar(labels, subset_counts["count"], color=colors, alpha=0.8, width=0.55)
axes[1].set_title("Image Count by Dataset Subset")
axes[1].set_ylabel("Number of Images")
axes[1].grid(axis="y", alpha=0.25)
for bar in bars:
    yval = int(bar.get_height())
    axes[1].text(bar.get_x() + bar.get_width() / 2, yval + 50, f"{yval}", ha="center", va="bottom", fontsize=9)
axes[1].set_ylim(0, max(subset_counts["count"]) * 1.15)

plt.tight_layout()
dist_path = ASSETS_DIR / "eda_distributions.png"
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {dist_path}")

In [ ]:
def pick_representative_indices(df: pd.DataFrame, k: int = 2) -> list[int]:
    non_empty = df[~df["is_empty"]].sort_values("crack_ratio_pct")
    pos = [int(len(non_empty) * 0.3), int(len(non_empty) * 0.8)]
    return [int(non_empty.index[p]) for p in pos]


cubit_indices = pick_representative_indices(df_cubit, k=2)
dc_indices = pick_representative_indices(df_dc_train, k=2)

samples = [
    ("CUBIT-Seg", cubit_ds[cubit_indices[0]]),
    ("CUBIT-Seg", cubit_ds[cubit_indices[1]]),
    ("DeepCrack", dc_train[dc_indices[0]]),
    ("DeepCrack", dc_train[dc_indices[1]]),
]

fig, axes = plt.subplots(len(samples), 3, figsize=(11, 3.2 * len(samples)))
for row, (ds_name, sample) in enumerate(samples):
    img = sample["image"]
    mask = sample["mask"]
    overlay = img.copy()
    overlay[mask == 1] = [255, 50, 50]
    blended = (0.6 * img + 0.4 * overlay).astype(np.uint8)

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{ds_name} | {sample['sample_id']}")
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"Binary GT Mask (ratio={mask.mean()*100:.2f}%)")
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title("RGB + GT Mask Overlay")

    for col in range(3):
        axes[row, col].axis("off")

plt.tight_layout()
samples_path = ASSETS_DIR / "eda_samples.png"
plt.savefig(samples_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {samples_path}")

## Summary of Findings

| Dataset | Images | Empty Masks | Mean Crack Ratio (%) | Median Crack Ratio (%) | 95th Percentile Ratio (%) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **CUBIT-Seg** | 5,462 | 0 | 2.002% | 1.814% | 4.845% |
| **DeepCrack** | 537 | 0 | 3.537% | 2.802% | 9.467% |

1. **Severe Class Imbalance**: Foreground crack pixels occupy only **2.00%** of the image area on average in `CUBIT-Seg` (median **1.81%**) and **3.54%** in `DeepCrack`. Evaluation must rely on **Pixel F1 (Dice), IoU, and Boundary-Tolerant F1** rather than pixel accuracy.
2. **Domain Gap (Close-up Pavement vs. UAV Building Facade)**: `DeepCrack` exhibits higher contrast and larger crack area coverage than `CUBIT-Seg`, motivating contrast normalization (CLAHE) and morphological post-processing.